# Week 2 - AI Applications - Exercise 2: Apartment Predictor + LLM Workflow

Course-fit note: This Week 2 variant is specific to AI Applications and combines a numeric model with an LLM interaction layer.

Because this is Swiss housing data, write your prompts in German so town names such as `Zürich` match the dataset more reliably.

In this exercise, you first build and test logic in the notebook, then transfer the same functions into `app_student.py`, and promote to `app.py` for deployment.


## Learning Goals

- Load and use a saved `.pkl` model for numeric prediction
- Convert natural-language wishes into structured model input
- Build a thin LLM explanation layer around model output
- Prepare a clean transfer from notebook code to a Gradio app


In [ ]:
import json
import os
import pickle
import re
import urllib.error
from pathlib import Path

import numpy as np
import pandas as pd
from openai import OpenAI


In [ ]:
DATA_PATH = Path("bfs_municipality_and_tax_data.csv")
MODEL_PATH = Path("random_forest_regression.pkl")

LLM_API_KEY = os.getenv("LLM_API_KEY", "")
LLM_MODEL = os.getenv("LLM_MODEL", "")

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Missing file: {DATA_PATH}")

if not MODEL_PATH.exists():
    raise FileNotFoundError(f"Missing file: {MODEL_PATH}")

df_bfs_data = pd.read_csv(DATA_PATH)
df_bfs_data["tax_income"] = (
    df_bfs_data["tax_income"].astype(str).str.replace("'", "", regex=False).astype(float)
)

town_to_row = {str(row["bfs_name"]).lower(): row for _, row in df_bfs_data.iterrows()}
valid_towns = list(df_bfs_data["bfs_name"].sort_values().unique())

# Optional: create an OpenAI client if you use the OpenAI SDK for your TODOs.
# client = OpenAI(api_key=LLM_API_KEY) if LLM_API_KEY else None

print(f"Loaded towns: {len(valid_towns)}")
df_bfs_data.head(2)


## Step 1 - Load The Saved `.pkl` Model

Reuse the provided scikit-learn model file instead of training a new model in this notebook.


In [ ]:
# TODO: load the pickled model from MODEL_PATH
# with open(MODEL_PATH, "rb") as model_file:
#     model = pickle.load(model_file)
model = None

if model is None:
    print("TODO: load model before continuing")
else:
    print(f"Loaded model type: {type(model).__name__}")


## Step 2 - Town Matching Helper


In [ ]:
def match_town(user_town: str):
    """Return the canonical town name from the dataset, or None."""
    # TODO
    # 1) handle empty input
    # 2) exact lower-case match
    # 3) relaxed contains-match over valid_towns
    raise NotImplementedError


## Step 3 - Extract Preferences From Natural Language

Use an LLM call similar to Week 1 `llm_calls`
Write the user prompts in German because the dataset contains Swiss place names such as `Zürich`.

Helpful structure for this step:
- give the model a short system/developer instruction,
- tell it to return strict JSON only,
- name the three required keys exactly: `rooms`, `area_m2`, `town`,
- tell it to use numbers for `rooms` and `area_m2`.

Example user input:
`Ich suche eine 3.5-Zimmer-Wohnung mit etwa 85 m2 in Winterthur.`

Ideal JSON shape:
```json
{"rooms": 3.5, "area_m2": 85, "town": "Winterthur"}
```

After the LLM call, still validate in Python that all three values exist and that `town` can be matched with `match_town(...)`.


In [ ]:
def extract_preferences(user_text: str) -> dict:
    """Extract rooms, area_m2, and town from free text."""
    # TODO
    # LLM is mandatory for this step.
    # You may use any provider/model.
    # Good prompt idea:
    # - ask for strict JSON only
    # - require keys: rooms, area_m2, town
    # - require numeric values for rooms and area_m2
    # Then parse the JSON and validate the extracted values in Python.
    # Do not implement regex or template fallback logic.
    raise NotImplementedError


## Step 4 - Predict Monthly Rent

Use the loaded random forest model and the same seven input features shown in `apartment.ipynb` from the exercise where you created the model.


In [ ]:
def predict_apartment_price(rooms: float, area_m2: float, town: str) -> float:
    # TODO
    # Build the 7-feature input in this exact order:
    # [rooms, area_m2, pop, pop_dens, frg_pct, emp, tax_income]
    # Then call model.predict(...) and return the rounded CHF value.
    # Hint: the pickled scikit-learn model returns a 1D prediction array.
    raise NotImplementedError


## Step 5 - Generate A User-Friendly Explanation

This second LLM step should not predict the price again. The model prediction already exists. The LLM should only explain the result in simple language.

A good explanation prompt should include:
- the structured preferences,
- the predicted rent in CHF,
- a request for a short German answer,
- one uncertainty or limitation note.

Ideal output shape:
```json
{"answer": "Für eine 3.5-Zimmer-Wohnung in Winterthur schÃ¤tzt das Modell rund 2800 CHF pro Monat. Die Schätzung orientiert sich an Wohnfläche und Ortsmerkmalen. Eine Unsicherheit ist, dass Zustand, Lage im Ort und Ausstattungsstandard im Modell nicht direkt enthalten sind."}
```


In [ ]:
def generate_explanation(preferences: dict, prediction: float) -> str:
    # TODO
    # LLM is mandatory for this step.
    # You may use any provider/model.
    # Pass the already computed prediction into the prompt.
    # Ask for one short German explanation in JSON with key: answer
    # Include one uncertainty note in the response.
    # Do not implement template fallback logic.
    raise NotImplementedError


## Step 6 - End-To-End Pipeline

Suggested order inside `run_pipeline(...)`:
1. call `extract_preferences(...)`
2. call `predict_apartment_price(...)`
3. call `generate_explanation(...)`
4. return `(preferences, prediction, answer)`


In [ ]:
def run_pipeline(user_text: str):
    # TODO
    # Return: (preferences_dict, prediction_float, final_answer_text)
    raise NotImplementedError


## Step 7 - Test With Multiple Examples


In [ ]:
test_inputs = [
    "Ich suche eine 3.5-Zimmer-Wohnung mit 85 m2 in Winterthur.",
    "Ich suche 2 Zimmer und etwa 55 m2 in Kloten.",
    "Ich brauche eine 4-Zimmer-Wohnung mit rund 110 m2 in Zürich.",
]

# TODO: loop over test_inputs and print results from run_pipeline(...)


## Step 8 - Notebook To app_student.py Then app.py

Copy your final function implementations into `app_student.py`:
- `extract_preferences`
- `predict_apartment_price`
- `generate_explanation`
- `run_pipeline`

Keep the saved model path as `random_forest_regression.pkl`.

Use `NOTEBOOK_TO_APP.md` as the deployment checklist.


## Submission Requirements

1. Working numeric prediction flow in notebook using the provided `.pkl` model
2. Working free-text input parsing to structured fields (LLM required)
3. Extraction prompt designed so the LLM returns strict JSON with `rooms`, `area_m2`, and `town`
4. Test inputs and prompts written in German so Swiss town names match the dataset reliably
5. Response text generated with an LLM and including one uncertainty/limitation
6. Transfer of notebook logic into `app_student.py`, then promote to `app.py`
7. No fallback path: missing key/API errors must remain visible
8. Short reflection (3-5 sentences): strengths, limits, responsible use
